> ## SOLUTION / ANSWER KEY &mdash; Lab 6.11
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-11-guardrails-observability.ipynb`](../lab-11-guardrails-observability.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.11 &mdash; Guardrails & Observability

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Allow-list the tools the agent may call
- Validate tool input before it runs
- Record every step with a tracing callback (LangSmith/Langfuse-style)

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, routing and guardrails &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable.

**Reference:** [Module 6 slides &mdash; Observability & debugging](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-06-11"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Autonomy needs **guardrails** and **observability** (deck slides 15, 17): a **tool allow-list**,
**input validation**, a **recursion_limit** cap, and a **tracing callback** that logs every
(action, input, observation) so you can debug and audit. Real LangChain exposes callbacks via
`BaseCallbackHandler`; **LangSmith / Langfuse** capture full traces. Here you build the offline
versions (deterministic) and see the real callback interface in the optional cell.

In [ ]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """Do exact arithmetic on an expression."""
    return "(computed)"

ALLOWED = {"calculator", "lookup"}
print("allow-list:", ALLOWED, "| a tool the agent may call:", calculator.name in ALLOWED)

## Your Turn
Complete the allow-list check, the input validator, and the tracing callback, then run a guarded loop.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool

@tool
def calculator(expression: str) -> float:
    """Do exact arithmetic on an expression."""
    return safe_calc(expression)

@tool
def lookup(key: str) -> str:
    """Look up a known fact by key."""
    return {"capital of france": "Paris"}.get(key.lower().strip(), "unknown")

REGISTRY = {t.name: t for t in [calculator, lookup]}
ALLOWED = {"calculator", "lookup"}

class TracingCallback:
    def __init__(self):
        self.events = []
    def on_step(self, action, arg, observation):
        self.events.append((action, arg, observation))

def is_allowed(action):
    return action in ALLOWED

def validate(expr):
    ok_chars = set("0123456789+-*/(). ")
    return all(c in ok_chars for c in expr) and any(c.isdigit() for c in expr)

def guarded_run(steps, cb, max_iterations=5):
    # steps: list of (action, arg). Enforce the allow-list + step cap; trace each executed step.
    for i, (action, arg) in enumerate(steps):
        if i >= max_iterations:
            return {'stopped': 'max_iterations', 'trace': cb.events}
        if not is_allowed(action):
            return {'stopped': 'blocked_tool', 'trace': cb.events}
        obs = REGISTRY[action].invoke(arg)
        cb.on_step(action, arg, obs)
    return {'stopped': 'done', 'trace': cb.events}

try:
    cb = TracingCallback()
    good = guarded_run([('calculator', '2+2'), ('lookup', 'capital of france')], cb)
    print('good run:', good['stopped'], '| trace:', good['trace'])
    blocked = guarded_run([('delete_database', 'all')], TracingCallback())
    print('blocked :', blocked['stopped'])
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("allow-list permits a known tool", lambda: is_allowed("calculator") is True)
expect_true("allow-list blocks an unknown tool", lambda: is_allowed("delete_database") is False)
expect_true("validation accepts a real expression", lambda: validate("2 + 2*3") is True)
expect_true("validation rejects a dangerous input", lambda: validate("__import__('os')") is False)
expect_true("the callback records every executed step", lambda: len(guarded_run([("calculator", "2+2"), ("lookup", "capital of france")], TracingCallback())["trace"]) == 2)
expect_true("a disallowed tool is blocked, not run", lambda: guarded_run([("delete_database", "all")], TracingCallback())["stopped"] == "blocked_tool")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
See the REAL callback interface LangChain exposes (LangSmith / Langfuse capture full traces). This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
try:
    from langchain_core.callbacks import BaseCallbackHandler
    class PrintHandler(BaseCallbackHandler):
        def on_tool_end(self, output, **kw):
            print("tool ->", output)
    print("Real LangChain calls handlers like PrintHandler on every model/tool event.")
    print("Attach one with: agent.invoke(inputs, config={'callbacks': [PrintHandler()]}).")
    print("For full run traces: set LANGCHAIN_TRACING_V2=true + LANGCHAIN_API_KEY (LangSmith),")
    print("or run Langfuse (this course's stack) and attach its callback handler.")
except Exception as e:
    print("Install langchain-core to use real callbacks -- skipping:", type(e).__name__)
print("The guarded run above already traced every step offline.")

---
### Key takeaway
Allow-list + validation + recursion_limit + tracing turn an autonomous agent from a liability into something you can trust and debug. Day 5 goes deeper on responsible agents.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>